# Scripts to obtain results

The scripts are divided into three main parts according to the solver being used: 
- MUMPS
- SUPERLU_DIST
- PASTIX

Solver-specific options are set for all of these scripts.

If you want to use the `reorder()` function before calling the `solve_system()` function itself and "separate" the reordering this way, use the `-own_reordering true` option. Otherwise, use `-own_reordering false` and the reordering will be handled by the selected solver.

---

In [12]:
import os
import subprocess

In [ ]:
# data paths
mat_input_path = "../matrices/bin/"
mat_permuted_path = "../matrices/bin/permuted/"
vec_input_path = "../matrices/bin/vec/"

# global paths for runs
EXE = "../petsc/arch-linux-c-opt/bin/mpirun -n {nproc} ../scripts/experiment"
MAT_INPUT_PATH = "../matrices/bin"

## MUMPS runs

In [26]:
# paths
LOG_DIR = "./mumps"  # this is where the log files will be saved
os.makedirs(LOG_DIR, exist_ok=True)

# parameters
nproc = 4            # number of processes being used
solver = "mumps"     
pc_type = "lu"
ordering = "natural" # set this option usually along with -own_reordering true,
                     # otherwise internal solver reordering will be used

mat_files = [f for f in os.listdir(MAT_INPUT_PATH) if f.endswith(".bin")]

for mat_file in mat_files:
    mat_path = os.path.join(MAT_INPUT_PATH, mat_file)
    log_file = f"{ordering}_{os.path.splitext(mat_file)[0]}_{solver}_{nproc}procs.log"
    log_path = os.path.join(LOG_DIR, log_file)

    cmd = (
        f"{EXE.format(nproc=nproc)} "
        f"-mat_file {mat_path} "          # the file path to the matrix in .bin format
        f"-pc_type {pc_type} "            # the PC type
        f"-mat_solver_type {solver} "     # the solver to be used
        # f"-own_reordering true "         # true: enable reordering "separation", false: let solver handle reordering
        # f"-mat_ordering_type {ordering} " # the ordering to be used
        f"-log_view "                     # print PETSc profiling

        f"-mat_mumps_icntl_2 3 "            # output stream for diagnostic printing, statistics, and warning
        f"-mat_mumps_icntl_4 2 "          # level of printing (0 to 4)
        f"-mat_mumps_icntl_7 5 "          # computes a symmetric permutation in sequential analysis, 
                                          # 0=AMD, 2=AMF, 3=Scotch, 4=PORD, 5=Metis, 6=QAMD, and 7=auto
        f"-mat_mumps_icntl_28 2 "           # use 1 for sequential analysis and ICNTL(7) ordering, 
                                           # or 2 for parallel analysis and ICNTL(29) ordering
        f"-mat_mumps_icntl_29 2 "          # parallel ordering 1 = ptscotch, 2 = parmetis
    )

    print(f"Running {mat_file} → {log_file}")
    with open(log_path, "w") as f:
        subprocess.run(cmd, shell=True, stdout=f, stderr=subprocess.STDOUT)

print("======= COMPLETE =======")


Running 03_s_ex10hs.bin → natural_03_s_ex10hs_mumps_4procs.log
Running 01_s_ex2.bin → natural_01_s_ex2_mumps_4procs.log
Running 02_u_rbsa480.bin → natural_02_u_rbsa480_mumps_4procs.log
Running 07_s_bundle1.bin → natural_07_s_bundle1_mumps_4procs.log
Running 06_u_bayer03.bin → natural_06_u_bayer03_mumps_4procs.log
Running 05_u_mhd3200a.bin → natural_05_u_mhd3200a_mumps_4procs.log
Running 10_u_airfoil_2d.bin → natural_10_u_airfoil_2d_mumps_4procs.log
Running 09_u_powersim.bin → natural_09_u_powersim_mumps_4procs.log
Running 04_s_nasa4704.bin → natural_04_s_nasa4704_mumps_4procs.log
Running 08_s_gyro_k.bin → natural_08_s_gyro_k_mumps_4procs.log
======= COMPLETE =======


## SUPERLU_DIST runs

In [ ]:
# paths
LOG_DIR = "./superlu_dist"
os.makedirs(LOG_DIR, exist_ok=True)

# parameters
nproc = 4            # number of processes being used
solver = "superlu_dist"     
pc_type = "lu"
ordering = "natural" # set this option usually along with -own_reordering true,
                     # otherwise internal solver reordering will be used


mat_files = [f for f in os.listdir(MAT_INPUT_PATH) if f.endswith(".bin")]

for mat_file in mat_files:
    mat_path = os.path.join(MAT_INPUT_PATH, mat_file)
    log_file = f"{ordering}_{os.path.splitext(mat_file)[0]}_{solver}_{nproc}procs.log"
    log_path = os.path.join(LOG_DIR, log_file)

    cmd = (
        f"{EXE.format(nproc=nproc)} "
        f"-mat_file {mat_path} "          # the file path to the matrix in .bin format
        f"-pc_type {pc_type} "            # the PC type
        f"-mat_solver_type {solver} "     # the solver to be used
        # f"-own_reordering true "         # true: enable reordering "separation", false: let solver handle reordering
        # f"-mat_ordering_type {ordering} " # the ordering to be used
        f"-log_view "                     # print PETSc profiling
        
        f"-mat_superlu_dist_rowperm NOROWPERM "# <NOROWPERM,LargeDiag_MC64,LargeDiag_AWPM,MY_PERMR> - row permutation
        f"-mat_superlu_dist_colperm PARMETIS " # <NATURAL,MMD_AT_PLUS_A,MMD_ATA,METIS_AT_PLUS_A,PARMETIS> - column
        f"-mat_superlu_dist_printstat"         # print factorization information
    )

    print(f"Running {mat_file} → {log_file}")
    with open(log_path, "w") as f:
        subprocess.run(cmd, shell=True, stdout=f, stderr=subprocess.STDOUT)

print("======= COMPLETE =======")


Running 03_s_ex10hs.bin → natural_03_s_ex10hs_superlu_dist_4procs.log
Running 01_s_ex2.bin → natural_01_s_ex2_superlu_dist_4procs.log
Running 02_u_rbsa480.bin → natural_02_u_rbsa480_superlu_dist_4procs.log
Running 07_s_bundle1.bin → natural_07_s_bundle1_superlu_dist_4procs.log
Running 06_u_bayer03.bin → natural_06_u_bayer03_superlu_dist_4procs.log
Running 05_u_mhd3200a.bin → natural_05_u_mhd3200a_superlu_dist_4procs.log
Running 10_u_airfoil_2d.bin → natural_10_u_airfoil_2d_superlu_dist_4procs.log
Running 09_u_powersim.bin → natural_09_u_powersim_superlu_dist_4procs.log
Running 04_s_nasa4704.bin → natural_04_s_nasa4704_superlu_dist_4procs.log
Running 08_s_gyro_k.bin → natural_08_s_gyro_k_superlu_dist_4procs.log
======= COMPLETE =======


## PaStiX runs

In [28]:
# paths
LOG_DIR = "./pastix"
os.makedirs(LOG_DIR, exist_ok=True)

# parameters
nproc = 4            # number of processes being used
solver = "pastix"     
pc_type = "lu"
ordering = "natural" # set this option usually along with -own_reordering true,
                     # otherwise internal solver reordering will be used


mat_files = [f for f in os.listdir(MAT_INPUT_PATH) if f.endswith(".bin")]

for mat_file in mat_files:
    mat_path = os.path.join(MAT_INPUT_PATH, mat_file)
    log_file = f"{ordering}_{os.path.splitext(mat_file)[0]}_{solver}_{nproc}procs.log"
    log_path = os.path.join(LOG_DIR, log_file)

    cmd = (
        f"{EXE.format(nproc=nproc)} "
        f"-mat_file {mat_path} "          # the file path to the matrix in .bin format
        f"-pc_type {pc_type} "            # the PC type
        f"-mat_solver_type {solver} "     # the solver to be used
        # f"-own_reordering true "         # true: enable reordering "separation", false: let solver handle reordering
        # f"-mat_ordering_type {ordering} " # the ordering to be used
        f"-log_view "                     # print PETSc profiling

        f"-mat_pastix_verbose 1 "         # <0,1,2> - print level of information messages from PaStiX
        f"-mat_pastix_factorization 0 "     # <0,1,2,3,4> - Factorization algorithm (Cholesky, LDL^t, LU, LL^t, LDL^h)
        f"-mat_pastix_ordering 0 "          # <0,1> - Ordering (Scotch or Metis)
        f"-mat_pastix_thread_nbr 2 "        # Set the numbers of threads for each MPI process
    )

    print(f"Running {mat_file} → {log_file}")
    with open(log_path, "w") as f:
        subprocess.run(cmd, shell=True, stdout=f, stderr=subprocess.STDOUT)

print("======= COMPLETE =======")

Running 03_s_ex10hs.bin → natural_03_s_ex10hs_pastix_4procs.log
Running 01_s_ex2.bin → natural_01_s_ex2_pastix_4procs.log
Running 02_u_rbsa480.bin → natural_02_u_rbsa480_pastix_4procs.log
Running 07_s_bundle1.bin → natural_07_s_bundle1_pastix_4procs.log
Running 06_u_bayer03.bin → natural_06_u_bayer03_pastix_4procs.log
Running 05_u_mhd3200a.bin → natural_05_u_mhd3200a_pastix_4procs.log


KeyboardInterrupt: 